# Mapas do MapBiomas para a página "A floresta da sua infância"

Roda tudo no navegador, sem instalar nada. Clique em **Ambiente de execução > Executar tudo**.

1. Na primeira célula, troque `SEU-PROJETO` pelo ID do seu projeto do Earth Engine.
2. Quando pedir, autorize com a sua conta Google.
3. No fim, o arquivo `mapas.zip` é baixado. Envie para o Claude no projeto.

In [ ]:
PROJETO = 'SEU-PROJETO'  # ID do projeto do Google Cloud ligado ao Earth Engine
TESTE = False  # True gera só Eldorado do Carajás (1502954), para conferir antes

In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project=PROJETO)
print('Earth Engine pronto')

In [ ]:
!wget -q -O dados_pagina.json https://raw.githubusercontent.com/primottags/ria-floresta-viva/chute-3d/public/dados_pagina.json
import argparse, json, pathlib, struct, sys, time

import ee
import requests

ASSET = "projects/mapbiomas-public/assets/brazil/lulc/collection11/mapbiomas_brazil_collection11_coverage_v3"
ANOS = list(range(1985, 2026))
FLORESTA = [3, 4, 5, 6, 7, 49]
AGUA = [31, 33]
CORES = ["edde8e", "1f8d49", "2e7dd7"]  # resto, floresta, água
IBGE = "https://servicodados.ibge.gov.br/api/v3/malhas/municipios/{cod}?formato=application/vnd.geo%2Bjson&qualidade=intermediaria"


def tamanho_png(caminho):
    with open(caminho, "rb") as f:
        cab = f.read(24)
    return struct.unpack(">II", cab[16:24])


def geometria(cod):
    for tentativa in range(4):
        try:
            r = requests.get(IBGE.format(cod=cod), timeout=60)
            r.raise_for_status()
            gj = r.json()
            feats = gj.get("features") or [gj]
            return ee.Geometry(feats[0]["geometry"])
        except Exception as erro:  # rede instável: tenta de novo
            if tentativa == 3:
                raise
            time.sleep(3 * (tentativa + 1))


def tira(cod, mapa, dimensao):
    geom = geometria(cod)
    quadros = []
    for ano in ANOS:
        banda = mapa.select(f"classification_{ano}")
        classe = (banda.remap(FLORESTA + AGUA, [1] * len(FLORESTA) + [2] * len(AGUA), 0)
                  .clip(geom).rename("c"))
        quadros.append(classe.visualize(min=0, max=2, palette=CORES).set("ano", ano))
    colecao = ee.ImageCollection(quadros)
    url = colecao.getFilmstripThumbURL({"dimensions": dimensao, "region": geom.bounds(), "format": "png", "crs": "EPSG:3857"})
    return requests.get(url, timeout=300)




In [ ]:
mapa = ee.Image(ASSET)
dados = json.loads(pathlib.Path('dados_pagina.json').read_text(encoding='utf-8'))
codigos = [m[1] for m in dados['m']]
if TESTE:
    codigos = [1502954]
saida = pathlib.Path('mapas'); saida.mkdir(exist_ok=True)
indice_arq = saida / 'indice.json'
indice = json.loads(indice_arq.read_text()) if indice_arq.exists() else {}
falhas = []
for n, cod in enumerate(codigos, 1):
    arq = saida / f'{cod}.png'
    if arq.exists() and str(cod) in indice:
        continue
    try:
        r = tira(cod, mapa, 240)
        r.raise_for_status()
        arq.write_bytes(r.content)
        w, h = tamanho_png(arq)
        indice[str(cod)] = [w, h // len(ANOS)]
        indice_arq.write_text(json.dumps(indice, separators=(',', ':')))
        print(f'[{n}/{len(codigos)}] {cod} ok')
    except Exception as erro:
        falhas.append(cod)
        print(f'[{n}/{len(codigos)}] {cod} FALHOU: {erro}')
print(f'Pronto. {len(indice)} municípios com mapa. Falhas: {falhas or "nenhuma"}')

In [ ]:
from IPython.display import Image as Img
Img('mapas/1502954.png', width=240)  # confira: Eldorado do Carajás, 1985 em cima, 2025 embaixo

In [ ]:
!zip -qr mapas.zip mapas
from google.colab import files
files.download('mapas.zip')